# Contribution recovery · run-bound evidence

**Run All reads saved, hashed artifact inputs and renders only.** It cannot generate or fit. Legacy records are manifest/input-hash compatible only; historical smoke receipts are verified under their *recorded* identities, never recertified as current fits. All scientific conclusions remain **pending review for this exact run**.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str((Path.cwd() / '../../../..').resolve()))
from studies.recovery.reporting import read_report
from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
evidence = read_report(Path.cwd())
display(Markdown(f'**Evidence ID:** `{evidence["run_id"]}` · **Source:** `{evidence["kind"]}` · '
                 f'**Provenance:** {evidence["provenance"]}'))

## Configuration, provenance and coverage

The evidence ID hashes the resolved config, input files, report sources and environment. Changed inputs fail Run All. Eligible channels exclude exact known-zero effects; a zero denominator means **not represented**, not zero failures. Small groups (<10 evaluable channels) are flagged.

In [ ]:
display(Markdown('**Resolved configuration:**'))
display(pd.json_normalize(evidence['resolved_config']).T.rename(columns={0: 'resolved value'}))
display(pd.DataFrame([{'method': k, 'worlds': len({r['world_id'] for r in evidence['worlds'] if r.get('method', evidence['kind']) == k}), 'channels': sum(1 for c in evidence['channels'] if c.get('method', evidence['kind']) == k), 'eligible': sum(c['nonzero_effect'] for c in evidence['channels'] if c.get('method', evidence['kind']) == k)} for k in evidence['pattern_rates']]))
display(Markdown('**Optional exploration:** ' + str(evidence['capabilities'])))

## Noise, true parameters and variation

These are **oracle descriptive profiles**, not fit initialization and not predictors of recovery. Noise SD, baseline variation, media/control contribution SD, carryover alpha, MM multiplier and direct coefficients come from saved worlds.

In [ ]:
profiles = pd.json_normalize([{'world': r['world_id'], 'method': r.get('method', evidence['kind']), **r['profile']} for r in evidence['worlds']])
display(profiles[['world', 'method', 'realized_noise_sd', 'baseline_temporal_sd', 'carryover_alpha', 'mm_kappa_mult', 'media_beta', 'control_coefficient', 'media_true_temporal_sd', 'control_true_temporal_sd']].head(60))

## Computational diagnostics ≠ point recovery ≠ uncertainty

Invalid MCMC exploration is unresolved, *not* a point failure. Short-smoke ADVI finite optimization only is **not** a convergence certificate; R-hat/ESS do not apply to variational draws. OLS has no posterior uncertainty. Pointwise credible intervals describe saved posterior uncertainty, not causal or frequentist generalization to new datasets.

In [ ]:
diagnostics = pd.json_normalize([{'world': r['world_id'], 'method': r.get('method', evidence['kind']), 'status': r['computational_status'], **r['diagnostics']} for r in evidence['worlds']])
display(diagnostics.head(60))
recovery = pd.DataFrame([{'world': r['world_id'], 'method': r.get('method', evidence['kind']), 'status': r['computational_status'], 'media': r['recovery']['media_pass'], 'controls': r['recovery']['control_pass'], 'joint': r['recovery']['joint_pass'], 'total_media': (r['recovery']['total_media'] or {}).get('pass'), 'baseline': (r['recovery']['baseline'] or {}).get('pass'), 'uncertainty': r['uncertainty']['type']} for r in evidence['worlds']])
display(recovery.head(60))

## Pattern-specific **channel** failure rates

Rows count **the channels bearing the named pattern**, not all failures in the same world. Patterns overlap and upward/downward trends have distinct denominators. Total includes known-zero channels; eligible excludes known-zero; unresolved eligible channels do not enter the failure-rate denominator.

In [ ]:
for method, rates in evidence['pattern_rates'].items():
    display(Markdown('### ' + method))
    display(pd.DataFrame(rates)[['kind','pattern','total','eligible','known_zero','evaluable','failed','unresolved','failure_rate','status','small_count_warning']])
channel_table = pd.DataFrame(evidence['channels'])
display(channel_table.head(25))

## Selectable true/fitted trajectories

Select a world/method and a **nonzero** contribution. Separate Plotly panels show its contribution (and available 95% interval), raw media spend for media channels, and the baseline (always retained). Legends sit below the panels. Known-zero padded/null channels are hidden; absence is not a success.

In [ ]:
keys = sorted(evidence['trajectories'])
selection = widgets.Dropdown(options=keys, value=keys[0], description='World · method:', layout=widgets.Layout(width='450px'))
def _eligible(key):
    if key not in evidence['trajectories']:
        return ('Baseline',)
    t = evidence['trajectories'][key]
    truth = np.asarray(t['truth'])
    return tuple(name for i, name in enumerate(t['terms']) if i == 0 or np.linalg.norm(truth[:, i]) > 0)
initial = _eligible(selection.value)
channel = widgets.Dropdown(options=initial, value=initial[0], description='Channel:', layout=widgets.Layout(width='300px'))
def _update_channels(change=None):
    valid = _eligible(selection.value)
    wanted = channel.value if channel.value in valid else valid[0]
    channel.options = valid
    channel.value = wanted
selection.observe(_update_channels, names='value')
def _view(key, name):
    if key not in evidence['trajectories'] or name not in _eligible(key):
        return None  # transient selector update, not a report/plot error
    t = evidence['trajectories'][key]
    ix = t['terms'].index(name)
    week = t['weeks']
    truth, fit = np.asarray(t['truth']), np.asarray(t['fitted'])
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=(name + ' contribution', 'Raw media spend (if media)', 'Baseline contribution'))
    def line(row, values, label, dash='solid'):
        fig.add_trace(go.Scatter(x=week, y=values, name=label, mode='lines', line={'dash': dash}, legendgroup=label, showlegend=True), row=row, col=1)
    line(1, truth[:, ix], 'True ' + name)
    line(1, fit[:, ix], 'Fitted ' + name, 'dash')
    if t['lower95'] is not None:
        low, high = np.asarray(t['lower95'])[:, ix], np.asarray(t['upper95'])[:, ix]
        fig.add_trace(go.Scatter(x=week, y=low, line={'width':0}, showlegend=False, hoverinfo='skip'), row=1, col=1)
        fig.add_trace(go.Scatter(x=week, y=high, fill='tonexty', fillcolor='rgba(80,80,180,.17)', line={'width':0}, name='95% pointwise interval'), row=1, col=1)
    if name.startswith('C'):
        line(2, np.asarray(t['raw_spend'])[:, int(name[1:])-1], 'Raw spend ' + name)
    line(3, truth[:, 0], 'True baseline')
    line(3, fit[:, 0], 'Fitted baseline', 'dash')
    status = next((r['computational_status'] for r in evidence['worlds'] if str(r['world_id']) + (':' + r['method'] if 'method' in r else '') == key), 'unknown')
    fig.update_layout(height=800, template='plotly_white', legend={'orientation':'h', 'y':-.13, 'x':0}, title=key + ' · ' + name + ' · ' + status)
    display(fig)
    return fig
display(widgets.HBox([selection, channel]))
interactive = widgets.interactive_output(_view, {'key': selection, 'name': channel})
display(interactive)

## Interpretation · run-bound review

Pending until an explicit reviewer approves this exact run into a separate sealed sibling review artifact. A present but invalid/cross-run review FAILS CLOSED. Optional held-out-world logistic screening, t-SNE and correlated-pair analysis are **deferred/unsupported**.

In [ ]:
from studies.recovery.review_interpretation import attached_review
review = attached_review(Path.cwd())
if review is None:
    display(Markdown(Path('interpretation.md').read_text()))
else:
    display(Markdown('**Reviewed for run** `' + review['report_run_id'] + '` by **' + review['reviewer'] + '** · decision: approved · review `' + review['review_id'] + '`'))
    display(Markdown(review['interpretation']))